
# Cosmogrid full DV test with scale-aware file loading

This notebook mirrors the structure of the original notebook, but replaces the hard-coded
`np.load("../data/...bin4.npy")` lines with a small file-discovery layer.

Use the configuration cell to choose the wavelet scale label (for example `R1`) and/or the
corresponding smoothing scale `theta1_input` in arcmin. The loader tries:

1. an explicit user-provided filename mapping,
2. a direct pattern match with the scale label,
3. a pattern match with the numerical theta value,
4. a plain fallback file without a theta/scale suffix.

This is meant to make the notebook robust to filenames such as

- `all_kappas_halofit_nobaryons_bin4.npy`
- `all_kappas_halofit_nobaryons_bin4_theta40.0_ratio2.0.npy`
- `all_l1_norms_halofit_nobaryons_bin4_R1.npy`

Adjust the mapping in the configuration cell if your naming convention is slightly different.


In [7]:

%load_ext autoreload
%autoreload 2

import os
import glob
import re
from pathlib import Path as FilePath
from typing import Dict, Any, List, Optional

import numpy as np
import pyccl as ccl
import matplotlib.pyplot as plt
import pandas as pd
from scipy.interpolate import CubicSpline

from wale.CosmologyModel import *
from wale.CovarianceMatrix import *
from wale.InitializeVariables import *
from wale.VarianceCalculator import *
from wale.FilterFunctions import *
from wale.CommonUtils import *
from wale.CriticalPoints import *
from wale.ComputePDF import *
from wale.LoadSimulations import *


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:

# ------------------------------------------------------------------
# Configuration
# ------------------------------------------------------------------
DATA_DIR = FilePath("../data")
ID_ZBIN = 3                  # 0-based index; ID_ZBIN=3 means tomographic bin 4
BIN_NUMBER = ID_ZBIN + 1

def format_theta_tag(R1):
    """
    Convert R1 to the filename tag used in the files, e.g.
    40   -> 'theta40.0'
    40.0 -> 'theta40.0'
    60   -> 'theta60.0'
    """
    return f"theta{float(R1):.1f}"



def get_files_for_R1(R1, data_dir=DATA_DIR, bin_number=BIN_NUMBER):
    data_dir = Path(data_dir)
    theta_tag = format_theta_tag(R1)

    file_patterns = {
        "kappas":    f"all_kappas_*bin{bin_number}_{theta_tag}*.npy",
        "l1_norms":  f"all_l1_norms_*bin{bin_number}_{theta_tag}*.npy",
        "variances": f"all_variances_*bin{bin_number}_{theta_tag}*.npy",
        "params":    "selected_params_halofit.npy",   # usually not theta-dependent
    }

    resolved = {}

    for key, pattern in file_patterns.items():
        matches = sorted(data_dir.glob(pattern))

        if key == "params":
            params_path = data_dir / pattern
            if not params_path.exists():
                raise FileNotFoundError(f"Could not find parameter file: {params_path}")
            resolved[key] = params_path
            continue

        if len(matches) == 0:
            raise FileNotFoundError(
                f"No file found for '{key}' with R1={R1}, "
                f"bin{bin_number}, pattern='{pattern}'"
            )
        if len(matches) > 1:
            print(f"[warning] Multiple matches for {key}, taking first:")
            for m in matches:
                print("   ", m.name)

        resolved[key] = matches[0]

    return resolved

def load_data_for_R1(R1, data_dir=DATA_DIR, bin_number=BIN_NUMBER):
    files = get_files_for_R1(R1, data_dir=data_dir, bin_number=bin_number)

    loaded = {
        "kappas": np.load(files["kappas"], allow_pickle=True),
        "l1_norms": np.load(files["l1_norms"], allow_pickle=True),
        "variances": np.load(files["variances"], allow_pickle=True),
        "params": np.load(files["params"], allow_pickle=True),
        "files": files,
    }

    return loaded

R1 = 40

data = load_data_for_R1(R1, data_dir=DATA_DIR, bin_number=BIN_NUMBER)

all_kappas = data["kappas"]
all_l1_norms = data["l1_norms"]
all_variances = data["variances"]
selected_params = data["params"]

print("Loaded files:")
for k, v in data["files"].items():
    print(f"{k:10s} -> {v.name}")

TypeError: float() argument must be a string or a real number, not 'PosixPath'

In [11]:

# ------------------------------------------------------------------
# File discovery helpers
# ------------------------------------------------------------------
def _natural_sort_key(pathlike):
    s = str(pathlike)
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", s)]

def list_matching_files(data_dir: Path, stem: str, bin_number: int) -> List[Path]:
    pattern = f"{stem}*bin{bin_number}*.npy"
    return sorted(data_dir.glob(pattern), key=_natural_sort_key)

def choose_best_file(
    candidates: List[Path],
    *,
    explicit_name: Optional[str] = None,
    scale_label: Optional[str] = None,
    theta_value: Optional[float] = None,
) -> Path:
    if explicit_name:
        path = DATA_DIR / explicit_name
        if not path.exists():
            raise FileNotFoundError(f"Explicitly requested file not found: {path}")
        return path

    if not candidates:
        raise FileNotFoundError("No candidate files found.")

    # 1) Prefer files that explicitly contain the scale label.
    if scale_label is not None:
        label_hits = [p for p in candidates if scale_label.lower() in p.name.lower()]
        if len(label_hits) == 1:
            return label_hits[0]
        if len(label_hits) > 1:
            # Prefer shorter names if several match.
            return sorted(label_hits, key=lambda p: (len(p.name), p.name))[0]

    # 2) Prefer files that contain the theta value.
    if theta_value is not None:
        theta_tokens = {
            f"theta{theta_value}",
            f"theta{theta_value:.1f}",
            f"theta{int(theta_value)}",
        }
        theta_hits = [
            p for p in candidates
            if any(tok.lower() in p.name.lower().replace("_", "") for tok in theta_tokens)
            or any(tok.lower() in p.name.lower() for tok in theta_tokens)
        ]
        if len(theta_hits) == 1:
            return theta_hits[0]
        if len(theta_hits) > 1:
            return sorted(theta_hits, key=lambda p: (len(p.name), p.name))[0]

    # 3) Prefer the plain file without an explicit theta suffix.
    plain_hits = [p for p in candidates if "theta" not in p.name.lower() and "r" not in p.name.lower()]
    if len(plain_hits) == 1:
        return plain_hits[0]
    if len(plain_hits) > 1:
        return sorted(plain_hits, key=lambda p: (len(p.name), p.name))[0]

    # 4) Fall back to the first candidate in natural sort order.
    return candidates[0]

def resolve_data_files(
    data_dir: Path,
    *,
    bin_number: int,
    scale_label: str,
    theta1_input: float,
    explicit_map: Dict[str, Dict[str, Optional[str]]],
) -> Dict[str, Path]:
    theta_value = SCALE_TO_THETA.get(scale_label, theta1_input)
    explicit = explicit_map.get(scale_label, {})

    stems = {
        "kappas": "all_kappas_halofit",
        "l1_norms": "all_l1_norms_halofit",
        "variances": "all_variances",
    }

    resolved = {}
    for key, stem in stems.items():
        candidates = list_matching_files(data_dir, stem, bin_number)

        # Restrict further by common substrings when possible.
        if key == "kappas":
            candidates = [p for p in candidates if "kappa" in p.name.lower()]
        elif key == "l1_norms":
            candidates = [p for p in candidates if "l1" in p.name.lower()]
        elif key == "variances":
            candidates = [p for p in candidates if "variance" in p.name.lower()]

        resolved[key] = choose_best_file(
            candidates,
            explicit_name=explicit.get(key),
            scale_label=scale_label,
            theta_value=theta_value,
        )

    return resolved

resolved_files = resolve_data_files(
    DATA_DIR,
    bin_number=BIN_NUMBER,
    scale_label=SCALE_LABEL,
    theta1_input=theta1_input,
    explicit_map=EXPLICIT_SCALE_FILE_MAP,
)

for k, v in resolved_files.items():
    print(f"{k:10s} -> {v.name}")


kappas     -> all_kappas_halofit_nobaryons_bin4_theta40.0_ratio2.0.npy
l1_norms   -> all_l1_norms_halofit_nobaryons_bin4_theta40.0_ratio2.0.npy
variances  -> all_variances_fiducial_nobaryons_bin4_theta40.0_ratio2.0.npy


In [ ]:

# ------------------------------------------------------------------
# Load cosmology table and resolved data products
# ------------------------------------------------------------------
param_file = DATA_DIR / "selected_params_halofit_unique.npy"
if not param_file.exists():
    param_file = DATA_DIR / "selected_params_halofit.npy"

param_table = np.load(param_file, allow_pickle=True)
param_table

PARAM_NAMES = [r"$\Omega_{m}$", r"$\sigma_8$", r"$w_0$", r"$H_0$", r"$n_s$", r"$\Omega_b$"]

def load_cosmology_from_param_table(param_file: str, cosmo_index: int = 0) -> Dict[str, Any]:
    data = np.load(param_file, allow_pickle=True)

    if hasattr(data, "dtype") and data.dtype.names:
        def get_field(name_candidates):
            for nm in name_candidates:
                if nm in data.dtype.names:
                    return data[nm][cosmo_index] if data.ndim > 0 else data[nm].item()
            return None

        candidates = {
            "Omega_m": [r"$\\Omega_{m}$", "Omega_m", "Om", "omega_m", "omegam"],
            "sigma8": [r"$\\sigma_8$", "sigma8", "s8"],
            "w0": [r"$w_0$", "w0", "w"],
            "H0": [r"$H_0$", "H0", "h0"],
            "ns": [r"$n_s$", "ns"],
            "Omega_b": [r"$\\Omega_b$", "Omega_b", "Ob", "omegab"],
        }

        Omega_m = get_field(candidates["Omega_m"])
        sigma8 = get_field(candidates["sigma8"])
        w0 = get_field(candidates["w0"])
        H0 = get_field(candidates["H0"])
        ns = get_field(candidates["ns"])
        Omega_b = get_field(candidates["Omega_b"])
    else:
        arr = np.asarray(data)
        if arr.ndim == 1 and arr.size == 6:
            row = arr
        elif arr.ndim == 2 and arr.shape[1] >= 6:
            row = arr[cosmo_index]
        else:
            raise ValueError(f"Unsupported parameter table shape: {arr.shape}")

        Omega_m, sigma8, w0, H0, ns, Omega_b = row[:6]

    h = H0 / 100.0
    return {
        "Om": float(Omega_m),
        "sigma8": float(sigma8),
        "w": float(w0),
        "H0": float(H0),
        "h": float(h),
        "ns": float(ns),
        "Ob": float(Omega_b),
        "Oc": float(Omega_m - Omega_b),
        "wa": 0.0,
    }

cosmogrid_kappa = np.load(resolved_files["kappas"])
cosmogrid_l1norms = np.load(resolved_files["l1_norms"])
cosmogrid_variances = np.load(resolved_files["variances"])
cosmogrid_params = np.load(param_file, allow_pickle=True)

cosmogrid_snr = cosmogrid_kappa / np.sqrt(cosmogrid_variances[:, np.newaxis])

print("Loaded:")
print("  kappa      :", resolved_files["kappas"].name, cosmogrid_kappa.shape)
print("  l1_norms   :", resolved_files["l1_norms"].name, cosmogrid_l1norms.shape)
print("  variances  :", resolved_files["variances"].name, cosmogrid_variances.shape)
print("  parameters :", param_file.name, cosmogrid_params.shape)

nz_file = DATA_DIR / "nz" / f"nz_stage3_{ID_ZBIN + 1}_GRID.txt"
print(f"Using n(z) file: {nz_file}")

filter_type = "tophat"
invalid_index = []
predictionl1 = []
predictionl1snr = []
cosmogrid = True


In [ ]:

fig, ax = plt.subplots(1, 2, figsize=(12, 6))

for cosmoindex in range(0, 1):
    print(f"\n--- Cosmology index: {cosmoindex} ---")

    pars = load_cosmology_from_param_table(str(param_file), cosmo_index=cosmoindex)
    Om     = pars["Om"]
    sigma8 = pars["sigma8"]
    w      = pars["w"]
    h      = pars["h"]
    ns     = pars["ns"]
    Ob     = pars["Ob"]
    Oc     = pars["Oc"]
    wa     = pars["wa"]

    print(f"Ωm={Om:.4f}  σ8={sigma8:.4f}  w0={w:.3f}  h={h:.4f}  ns={ns:.4f}  Ωb={Ob:.4f}")

    theta1 = theta1_input

    variables = InitialiseVariables(
        h=h, Oc=Oc, Ob=Ob, w=w, wa=wa, sigma8=sigma8,
        dk=0.005, kmin=1e-2, kmax=0.5,
        nz_file=str(nz_file), variability=False,
        theta1=theta1, nplanes=10, ns=ns,
    )

    variance = Variance(variables.cosmo, filter_type=filter_type, pk=variables.cosmo.pnl)

    variables.sigmasq_map = float(np.sum(
        variables.dchi * (variables.lensingweights ** 2) * np.array([
            float(variance.get_sig_slice(z, chi * variables.theta1_radian, chi * variables.theta2_radian))
            for z, chi in zip(variables.redshifts, variables.chis)
        ])
    ))

    variables.recal_value = variables.sigmasq_map / cosmogrid_variances[cosmoindex]

    smallest_positive, largest_negative = find_critical_points_for_cosmo(
        variables, variance, ngrid_critical=25,
        plot=False, min_z=0, max_z=5,
    )
    if smallest_positive is not None and largest_negative is not None:
        variables.lambdas = np.linspace(largest_negative, smallest_positive, 20)

    kappa_grid = cosmogrid_kappa[cosmoindex]
    computed_PDF = computePDF(variables, variance, plot_scgf=False, kappa=kappa_grid)
    pdf_vals = np.array(computed_PDF.pdf_values)
    theory_l1_kappa = np.abs(kappa_grid) * pdf_vals

    sigma_cosmogrid = float(np.sqrt(cosmogrid_variances[cosmoindex]))
    snr_grid = kappa_grid / sigma_cosmogrid
    theory_l1_snr = sigma_cosmogrid * theory_l1_kappa

    ax[0].plot(snr_grid, theory_l1_snr, "r", alpha=0.8)
    ax[0].plot(cosmogrid_snr[cosmoindex], cosmogrid_l1norms[cosmoindex], "o", ms=1, color="blue")

    ax[1].plot(kappa_grid, theory_l1_kappa, "r", alpha=0.8)
    ax[1].plot(kappa_grid, cosmogrid_l1norms[cosmoindex], "o", ms=1, color="blue")

ax[0].set_xlabel(r"SNR $\kappa/\sigma$")
ax[0].set_ylabel(r"$|\kappa| P(\kappa)$")
ax[1].set_xlabel(r"$\kappa$")
ax[1].set_ylabel(r"$|\kappa| P(\kappa)$")
plt.tight_layout()
plt.show()


In [ ]:

def process_one_cosmology(cosmoindex):
    print(f"\n--- Cosmology index: {cosmoindex} ---")

    pars = load_cosmology_from_param_table(str(param_file), cosmo_index=cosmoindex)
    Om     = pars["Om"]
    sigma8 = pars["sigma8"]
    w      = pars["w"]
    h      = pars["h"]
    ns     = pars["ns"]
    Ob     = pars["Ob"]
    Oc     = pars["Oc"]
    wa     = pars["wa"]

    theta1 = theta1_input

    variables = InitialiseVariables(
        h=h, Oc=Oc, Ob=Ob, w=w, wa=wa, sigma8=sigma8,
        dk=0.005, kmin=1e-2, kmax=0.5,
        nz_file=str(nz_file), variability=False,
        theta1=theta1, nplanes=10, ns=ns,
    )

    variance = Variance(variables.cosmo, filter_type=filter_type, pk=variables.cosmo.pnl)

    variables.sigmasq_map = float(np.sum(
        variables.dchi * (variables.lensingweights ** 2) * np.array([
            float(variance.get_sig_slice(z, chi * variables.theta1_radian, chi * variables.theta2_radian))
            for z, chi in zip(variables.redshifts, variables.chis)
        ])
    ))

    sigma_cosmogrid = float(np.sqrt(cosmogrid_variances[cosmoindex]))
    variables.recal_value = variables.sigmasq_map / cosmogrid_variances[cosmoindex]

    smallest_positive, largest_negative = find_critical_points_for_cosmo(
        variables, variance, ngrid_critical=25,
        plot=False, min_z=0, max_z=5,
    )

    if smallest_positive is None or largest_negative is None:
        return {"valid": False, "index": cosmoindex}

    variables.lambdas = np.linspace(largest_negative, smallest_positive, 20)

    kappa_vals = cosmogrid_kappa[cosmoindex]
    computed_PDF = computePDF(variables, variance, plot_scgf=False, kappa=kappa_vals)
    pdf_vals = np.array(computed_PDF.pdf_values)

    theory_l1_kappa = np.abs(kappa_vals) * pdf_vals
    snr = kappa_vals / sigma_cosmogrid
    prediction_l1_snr = sigma_cosmogrid * theory_l1_kappa

    return {
        "valid": True,
        "index": cosmoindex,
        "kappa_vals": kappa_vals,
        "snr": snr,
        "sigma_cosmogrid": sigma_cosmogrid,
        "theory_l1_kappa": theory_l1_kappa,
        "prediction_l1_snr": prediction_l1_snr,
    }


In [ ]:

from joblib import Parallel, delayed

n_to_run = min(100, len(cosmogrid_params))
results = Parallel(n_jobs=4, prefer="threads", verbose=10)(
    delayed(process_one_cosmology)(i) for i in range(n_to_run)
)


In [ ]:

valid_results   = [r for r in results if r["valid"]]
invalid_indices = [r["index"] for r in results if not r["valid"]]
print(f"{len(valid_results)} valid, {len(invalid_indices)} invalid: {invalid_indices}")

fig, ax = plt.subplots(1, 2, figsize=(12, 6))

for r in valid_results:
    idx            = r["index"]
    kappa_vals     = r["kappa_vals"]
    snr            = r["snr"]
    sigma_cg       = r["sigma_cosmogrid"]
    pred_l1_snr    = r["prediction_l1_snr"]
    pred_l1_kappa  = r["theory_l1_kappa"]

    ax[0].plot(snr, pred_l1_snr, "r", alpha=0.6)
    ax[0].plot(cosmogrid_snr[idx], cosmogrid_l1norms[idx], "o", ms=1, color="blue")
    residual_snr = (pred_l1_snr - cosmogrid_l1norms[idx]) / cosmogrid_l1norms[idx]
    mask_snr = (snr > -2) & (snr < 2)
    ax[0].plot(snr[mask_snr][::2], residual_snr[mask_snr][::2], "x", color="green", ms=3)

    ax[1].plot(kappa_vals, pred_l1_kappa, "r", alpha=0.6)
    ax[1].plot(kappa_vals, cosmogrid_l1norms[idx], "o", ms=1, color="blue")
    residual_l1 = (pred_l1_kappa - cosmogrid_l1norms[idx]) / cosmogrid_l1norms[idx]
    mask_l1 = (kappa_vals > -2 * sigma_cg) & (kappa_vals < 2 * sigma_cg)
    ax[1].plot(kappa_vals[mask_l1][::2], residual_l1[mask_l1][::2], "x", color="green", ms=3)

ax[0].axhline(0, color="gray", linestyle="dashed")
ax[0].set_xlabel(r"SNR $\kappa/\sigma$")
ax[0].set_ylabel(r"$|\kappa| P(\kappa)$")
ax[0].set_xlim(-6, 6)
ax[0].set_ylim(-0.05, 0.3)

ax[1].axhline(0, color="gray", linestyle="dashed")
ax[1].set_xlabel(r"$\kappa$")
ax[1].set_ylabel(r"$|\kappa| P(\kappa)$")
ax[1].set_xlim(-0.015, 0.015)
ax[1].set_ylim(-0.05, 0.3)

plt.tight_layout()
plt.show()
